# Unified Modeling Pipeline (Segment-wise)

This notebook evaluates linear, tree-based, and SVR models across four tea market segments using chronological 5-fold cross-validation.

Segments:
- High Grown
- Low Grown
- Off-Grade
- Dust

Metrics per model x segment:
- RMSE
- MAE
- MAPE
- R2

In [30]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

SEED = 42
np.random.seed(SEED)

print('xgboost available: True')
print('lightgbm available: True')

xgboost available: True
lightgbm available: True


In [24]:
# Resolve processed input file across both casing conventions used in the repo.
ROOT = Path.cwd().resolve().parent.parent  # from src/processing -> project root
CANDIDATE_PATHS = [
    ROOT / 'data' / 'processed' / 'tea_preprocessed.csv',
    ROOT / 'data' / 'Processed' / 'tea_preprocessed.csv',
]

DATA_PATH = next((p for p in CANDIDATE_PATHS if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(f'Could not find tea_preprocessed.csv in: {CANDIDATE_PATHS}')

df = pd.read_csv(DATA_PATH)

# Keep rows with target available
if 'has_price_target' in df.columns:
    df = df[df['has_price_target'] == 1].copy()

print('Loaded:', DATA_PATH)
print('Data shape:', df.shape)
display(df.head(3))

Loaded: C:\Users\senil\Documents\GitHub\data-analysis-for-tea-industry\data\processed\tea_preprocessed.csv
Data shape: (2886, 182)


,sale_id,sale_number,sale_year,sale_month,sale_date_raw,table_source,elevation,category_type,grade,tier,...,all_regions__avg_precipitation,is_production_known,price_mid_usd,sale_month_enc,tier_enc,elevation_enc,category_type_enc,table_source_enc,price_mid_lkr_log,has_price_target
0,SALE_34_2025,34,2025.0,September,01ST/02ND September 2025,04_high_grown,high_grown,high_grown,bop,NaN,...,39.85,0,4.156889,9.0,0,3,1,0,7.123673,1
1,SALE_34_2025,34,2025.0,September,01ST/02ND September 2025,04_high_grown,high_grown,high_grown,bopf,NaN,...,39.85,0,4.324506,9.0,0,3,1,0,7.163172,1
2,SALE_34_2025,34,2025.0,September,01ST/02ND September 2025,04_high_grown,high_grown,high_grown,pekoe_fbop,NaN,...,39.85,0,4.089842,9.0,0,3,1,0,7.107425,1


In [25]:
# Segment filters aligned with existing project conventions.
# Supports both naming styles, e.g. 04_high_grown and 04_high_grown_prices.
ts = df['table_source'].astype(str).str.strip().str.lower()
ct = df['category_type'].astype(str).str.strip().str.lower()

SEGMENT_FILTERS = {
    'High Grown': ts.isin(['04_high_grown', '04_high_grown_prices']),
    'Low Grown': ts.isin(['05_low_grown', '05_low_grown_prices']),
    'Off-Grade': ts.isin(['06_offgrade_dust', '06_offgrade_dust_prices']) & (ct == 'off_grade'),
    'Dust': ts.isin(['06_offgrade_dust', '06_offgrade_dust_prices']) & (ct == 'dust'),
}

EXCLUDE_COLS = {
    'sale_id', 'sale_date_raw', 'sale_month', 'table_source', 'category_type', 'grade', 'tier', 'category',
    'price_mid_lkr', 'price_mid_lkr_log', 'has_price_target',
    'price_lo_lkr', 'price_hi_lkr', 'price_range_lkr'
}

TARGET = 'price_mid_lkr_log'
RANK_COL = 'sale_rank'

segment_data = {}
for seg, mask in SEGMENT_FILTERS.items():
    sdf = df[mask].copy()

    # Only use numeric predictors to avoid model failures on raw text/categorical columns.
    numeric_cols = [c for c in sdf.columns if pd.api.types.is_numeric_dtype(sdf[c])]
    feature_cols = [c for c in numeric_cols if c not in EXCLUDE_COLS and c != TARGET]

    if RANK_COL in sdf.columns:
        sdf = sdf.sort_values(RANK_COL).reset_index(drop=True)

    segment_data[seg] = (sdf, feature_cols)

    # Quick sanity diagnostics.
    raw_candidate_cols = [c for c in sdf.columns if c not in EXCLUDE_COLS and c != TARGET]
    non_numeric_count = len([c for c in raw_candidate_cols if c not in feature_cols])
    print(f"{seg:<10} rows={len(sdf):4d} features={len(feature_cols):3d} dropped_non_numeric={non_numeric_count}")

High Grown rows= 516 features=167 dropped_non_numeric=1
Low Grown  rows=1348 features=167 dropped_non_numeric=1
Off-Grade  rows= 499 features=167 dropped_non_numeric=1
Dust       rows= 523 features=167 dropped_non_numeric=1


In [31]:
def build_model_registry(seed=42):
    models = {
        'Ridge': Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale', StandardScaler()),
            ('model', Ridge(alpha=1.0, random_state=seed))
        ]),
        'Lasso': Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale', StandardScaler()),
            ('model', Lasso(alpha=0.001, random_state=seed, max_iter=10000))
        ]),
        'ElasticNet': Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale', StandardScaler()),
            ('model', ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=seed, max_iter=10000))
        ]),
        'Random Forest': Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('model', RandomForestRegressor(n_estimators=400, min_samples_leaf=3, random_state=seed, n_jobs=-1))
        ]),
        'Gradient Boosting': Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('model', GradientBoostingRegressor(random_state=seed))
        ]),
        'SVR (RBF)': Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale', StandardScaler()),
            ('model', SVR(kernel='rbf', C=10.0, epsilon=0.05, gamma='scale'))
        ]),
        'XGBoost': Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('model', XGBRegressor(
                n_estimators=400, learning_rate=0.05, max_depth=5,
                subsample=0.8, colsample_bytree=0.8, random_state=seed, n_jobs=-1
            ))
        ]),
        'LightGBM': Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('model', LGBMRegressor(
                n_estimators=400, learning_rate=0.05, num_leaves=31, random_state=seed
            ))
        ]),
    }

    return models

In [27]:
def compute_metrics(y_true_log, y_pred_log):
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)

    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    mape = float(np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100)
    r2 = float(r2_score(y_true_log, y_pred_log))
    return rmse, mae, mape, r2


def run_timeseries_cv(sdf, feature_cols, model_name, model_obj, target=TARGET, k=5):
    X = sdf[feature_cols].copy()
    y = sdf[target].copy()

    tscv = TimeSeriesSplit(n_splits=k)
    fold_rows = []

    for fold, (tr_idx, te_idx) in enumerate(tscv.split(X), start=1):
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

        model_obj.fit(X_tr, y_tr)
        pred_log = model_obj.predict(X_te)

        rmse, mae, mape, r2 = compute_metrics(y_te.values, pred_log)
        fold_rows.append({
            'Model': model_name, 'Fold': fold,
            'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'R2': r2,
            'n_train': len(tr_idx), 'n_test': len(te_idx)
        })

    return pd.DataFrame(fold_rows)

In [28]:
models = build_model_registry(seed=SEED)
all_folds = []

for seg_name, (sdf, feature_cols) in segment_data.items():
    print(f'\nRunning segment: {seg_name} | rows={len(sdf)} | features={len(feature_cols)}')

    # Need enough rows for 5 folds in TimeSeriesSplit.
    if len(sdf) < 20:
        print(f'  Skipped {seg_name}: not enough rows for stable 5-fold CV.')
        continue

    for model_name, model_obj in models.items():
        try:
            fold_df = run_timeseries_cv(sdf, feature_cols, model_name, model_obj, target=TARGET, k=5)
            fold_df.insert(0, 'Segment', seg_name)
            all_folds.append(fold_df)
            print(f'  Done: {model_name}')
        except Exception as ex:
            print(f'  Failed: {model_name} -> {ex}')

results_folds = pd.concat(all_folds, ignore_index=True) if all_folds else pd.DataFrame()
print('\nFold-level result shape:', results_folds.shape)
display(results_folds.head(10))


Running segment: High Grown | rows=516 | features=167
  Done: Ridge
  Done: Lasso
  Done: ElasticNet
  Done: Random Forest
  Done: Gradient Boosting
  Done: SVR (RBF)
  Done: XGBoost
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000359 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 638
[LightGBM] [Info] Number of data points in the train set: 86, number of used features: 133
[LightGBM] [Info] Start training from score 7.074904
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits

,Segment,Model,Fold,RMSE,MAE,MAPE,R2,n_train,n_test
0,High Grown,Ridge,1,55.272531,32.430451,5.307137,0.905062,86,86
1,High Grown,Ridge,2,52.237663,47.129621,4.481586,0.957837,172,86
2,High Grown,Ridge,3,61.199622,46.182430,4.333083,0.958189,258,86
3,High Grown,Ridge,4,296.402924,182.380538,16.394893,-1.094182,344,86
4,High Grown,Ridge,5,91.519261,63.032396,5.475195,0.961526,430,86
5,High Grown,Lasso,1,54.732935,34.814861,5.442863,0.909412,86,86
6,High Grown,Lasso,2,52.348666,46.692285,4.428448,0.958302,172,86
7,High Grown,Lasso,3,59.418297,43.552608,4.064711,0.962640,258,86
8,High Grown,Lasso,4,52.680786,45.467186,4.224227,0.962410,344,86
9,High Grown,Lasso,5,91.312213,59.890038,5.306666,0.962374,430,86


In [29]:
if results_folds.empty:
    print('No CV results generated.')
else:
    summary = (
        results_folds
        .groupby(['Segment', 'Model'], as_index=False)
        .agg(
            RMSE=('RMSE', 'mean'),
            MAE=('MAE', 'mean'),
            MAPE=('MAPE', 'mean'),
            R2=('R2', 'mean')
        )
        .sort_values(['Segment', 'RMSE'])
        .reset_index(drop=True)
    )

    print('Unified CV summary (mean over 5 folds):')
    display(summary)

    out_dir = ROOT / 'data' / 'processed'
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / 'unified_model_cv_results.csv'
    summary.to_csv(out_path, index=False)
    print(f'Saved summary to: {out_path}')

Unified CV summary (mean over 5 folds):


,Segment,Model,RMSE,MAE,MAPE,R2
0,Dust,Gradient Boosting,16.278658,10.371166,1.187690,0.983693
1,Dust,XGBoost,22.448408,14.613442,1.661383,0.974597
2,Dust,Random Forest,24.890406,15.246789,1.870700,0.964529
3,Dust,Lasso,32.140731,24.959597,2.749934,0.968933
4,Dust,ElasticNet,32.539076,24.974138,2.752542,0.968492
5,Dust,Ridge,35.195669,26.354051,2.911041,0.964145
6,Dust,LightGBM,40.891297,24.650455,3.139017,0.916865
7,Dust,SVR (RBF),253.421985,204.598844,19.885741,-0.456711
8,High Grown,Gradient Boosting,22.744785,13.999428,1.609281,0.984881
9,High Grown,XGBoost,28.498770,18.280348,2.073675,0.979671


Saved summary to: C:\Users\senil\Documents\GitHub\data-analysis-for-tea-industry\data\processed\unified_model_cv_results.csv
